# 02 — Parametric Geometry Explorer

**nEUROn v2** — BWB Flying Wing with integrated center body

This notebook visualizes the 30-variable parametric model:
1. Default BWB configuration (2-wing AeroSandbox model)
2. Center-body airfoil sections (thick NACA-based)
3. Outer wing Kulfan airfoils
4. C0 continuity verification at the blend station
5. Internal volume computation
6. EDF duct fit verification
7. Design space bounds overview

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

from src.parameterization.design_variables import BWBParams, BOUNDS, VAR_NAMES, N_VARS
from src.parameterization.bwb_aircraft import (
    build_airplane, build_body_airfoil, build_kulfan_airfoil_at_station,
    compute_wing_area, compute_aspect_ratio, compute_mac,
    compute_internal_volume, estimate_structural_mass,
    OUTER_WING_STATIONS, N_OUTER_SEGMENTS, BODY_STATIONS,
)
from src.propulsion.edf_model import EDF_70MM, duct_fits_in_body, min_body_tc_for_duct

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Default Configuration

In [ ]:
params = BWBParams()

print(f"═══ Design Variables ({N_VARS} total) ═══")
print(f"\nPlanform:")
print(f"  Half span        : {params.half_span:.3f} m  (full: {2*params.half_span:.2f} m)")
print(f"  Wing root chord  : {params.wing_root_chord:.3f} m")
print(f"  Taper ratio      : {params.taper_ratio}")
print(f"  LE sweep         : {params.le_sweep_deg}°")
print(f"\nCenter Body:")
print(f"  Body chord ratio : {params.body_chord_ratio} → chord = {params.body_root_chord:.3f} m")
print(f"  Body half-width  : {params.body_halfwidth:.3f} m")
print(f"  Body t/c root    : {params.body_tc_root:.1%}")
print(f"  Body camber      : {params.body_camber}")
print(f"  Body reflex      : {params.body_reflex}")
print(f"  Body twist       : {params.body_twist}°")
print(f"  Body sweep       : {params.body_sweep_deg}° (wing + {params.body_sweep_delta}°)")
print(f"  Body LE droop    : {params.body_le_droop}")
print(f"\nDerived Geometry:")
print(f"  Outer half-span  : {params.outer_half_span:.3f} m")
print(f"  Tip chord        : {params.tip_chord:.3f} m")
print(f"  Wing area        : {compute_wing_area(params):.4f} m²")
print(f"  Aspect ratio     : {compute_aspect_ratio(params):.2f}")
print(f"  MAC              : {compute_mac(params):.3f} m")
print(f"  Struct mass      : {estimate_structural_mass(params):.3f} kg")
print(f"  Internal volume  : {compute_internal_volume(params)*1e6:.0f} cm³")
print(f"  Duct fits        : {duct_fits_in_body(params.body_tc_root, params.body_root_chord, EDF_70MM)}")

## 2. Planform — Top View & Front View

The BWB has two distinct regions: center body (thick, more swept) and outer wing (tapered, Kulfan profiles).

In [ ]:
p = params
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Compute body positions ──
body_sweep_rad = np.radians(p.body_sweep_deg)
bw = p.body_halfwidth
body_chord_root = p.body_root_chord
body_chord_tip = p.wing_root_chord

body_le = [[0, 0], [bw * 0.5 * np.tan(body_sweep_rad), bw * 0.5],
           [bw * np.tan(body_sweep_rad), bw]]
body_te = [[body_chord_root, 0],
           [(body_chord_root + body_chord_tip) / 2 + bw * 0.5 * np.tan(body_sweep_rad), bw * 0.5],
           [body_chord_tip + bw * np.tan(body_sweep_rad), bw]]

# ── Compute outer wing positions ──
wing_sweep_rad = np.radians(p.le_sweep_deg)
outer_span = p.outer_half_span
root_chord = p.wing_root_chord
tip_chord = p.tip_chord
dihedrals = [p.dihedral_0, p.dihedral_1, p.dihedral_2, p.dihedral_3, p.dihedral_tip]

x_blend = bw * np.tan(body_sweep_rad)
wing_le = [[x_blend, bw, 0]]
for i in range(N_OUTER_SEGMENTS):
    prev = wing_le[-1]
    dy_seg = outer_span * (OUTER_WING_STATIONS[i+1] - OUTER_WING_STATIONS[i])
    dih_rad = np.radians(dihedrals[i])
    dx = dy_seg * np.tan(wing_sweep_rad)
    dy = dy_seg * np.cos(dih_rad)
    dz = dy_seg * np.sin(dih_rad)
    wing_le.append([prev[0]+dx, prev[1]+dy, prev[2]+dz])

chords = [root_chord + f*(tip_chord-root_chord) for f in OUTER_WING_STATIONS]
wing_te = [[le[0]+c, le[1], le[2] if len(le)>2 else 0] for le, c in zip(wing_le, chords)]

# ═══ TOP VIEW ═══
ax = axes[0]
for sign in [1, -1]:
    # Body
    by = [pt[1]*sign for pt in body_le]
    bx_le = [pt[0] for pt in body_le]
    bx_te = [pt[0] for pt in body_te]
    ax.fill_betweenx(by, bx_le, bx_te, alpha=0.15, color="coral")
    ax.plot(bx_le, by, "r-", lw=2)
    ax.plot(bx_te, by, "r-", lw=2)

    # Outer wing
    wy = [pt[1]*sign for pt in wing_le]
    wx_le = [pt[0] for pt in wing_le]
    wx_te = [pt[0] for pt in wing_te]
    ax.fill_betweenx(wy, wx_le, wx_te, alpha=0.1, color="steelblue")
    ax.plot(wx_le, wy, "b-", lw=2)
    ax.plot(wx_te, wy, "b-", lw=2)
    ax.plot([wx_le[-1], wx_te[-1]], [wy[-1], wy[-1]], "b-", lw=2)

# Root closure
ax.plot([0, body_chord_root], [0, 0], "r-", lw=2)
ax.plot(p.body_root_chord * 0.30, 0, "go", ms=8, label="CG ref")
ax.set_xlabel("Chord [m]")
ax.set_ylabel("Span [m]")
ax.set_title(f"Top View — span={2*p.half_span:.2f}m, AR={compute_aspect_ratio(p):.1f}")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# ═══ FRONT VIEW ═══
ax2 = axes[1]
for sign in [1, -1]:
    # Body (flat at z=0)
    ax2.plot([0, bw*sign], [0, 0], "r-", lw=3, label="Body" if sign==1 else None)
    # Outer wing with dihedral
    wy = [pt[1]*sign for pt in wing_le]
    wz = [pt[2] if len(pt)>2 else 0 for pt in wing_le]
    ax2.plot(wy, wz, "b-", lw=2.5, label="Wing" if sign==1 else None)

ax2.plot(0, 0, "go", ms=8)
ax2.set_xlabel("Span [m]")
ax2.set_ylabel("Height [m]")
ax2.set_title(f"Front View — Gull Wing [{p.dihedral_0:.0f}°, {p.dihedral_1:.0f}°, "
              f"{p.dihedral_2:.0f}°, {p.dihedral_3:.0f}°, {p.dihedral_tip:.0f}°]")
ax2.set_aspect("equal")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Airfoil Sections

4 key sections: body center (thick), body blend/wing root, wing mid-span, wing tip.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)

sections = [
    ("Body Center (y=0)", build_body_airfoil(
        p.body_tc_root, p.body_camber, p.body_reflex, p.body_le_droop, "center"
    ), "coral"),
    ("Wing Root / Body Blend", build_kulfan_airfoil_at_station(p, 0.0, "root").to_airfoil(), "steelblue"),
    ("Wing 50% span", build_kulfan_airfoil_at_station(p, 0.5, "mid").to_airfoil(), "steelblue"),
    ("Wing Tip", build_kulfan_airfoil_at_station(p, 1.0, "tip").to_airfoil(), "steelblue"),
]

for ax, (title, af, color) in zip(axes.flat, sections):
    coords = af.coordinates
    ax.plot(coords[:, 0], coords[:, 1], color=color, lw=1.5)
    ax.fill(coords[:, 0], coords[:, 1], alpha=0.1, color=color)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("y/c")

axes[1, 0].set_xlabel("x/c")
axes[1, 1].set_xlabel("x/c")

plt.tight_layout()
plt.show()

## 4. AeroSandbox Model — Build & Verify

Build the full 2-wing AeroSandbox model and verify the structure.

In [ ]:
airplane = build_airplane(params)

print(f"Airplane: {airplane.name}")
print(f"Wings: {len(airplane.wings)}")
for wing in airplane.wings:
    print(f"\n  {wing.name}:")
    print(f"    Symmetric: {wing.symmetric}")
    print(f"    Sections: {len(wing.xsecs)}")
    for j, xsec in enumerate(wing.xsecs):
        print(f"      [{j}] LE={[f'{v:.4f}' for v in xsec.xyz_le]}, "
              f"chord={xsec.chord:.4f}m, twist={xsec.twist:.1f}°, "
              f"airfoil={xsec.airfoil.name}")

# C0 continuity check
body_tip = airplane.wings[0].xsecs[-1]  # last body section
wing_root = airplane.wings[1].xsecs[0]  # first wing section
print(f"\n═══ C0 Continuity at Blend ═══")
print(f"  Body tip LE  : {body_tip.xyz_le}")
print(f"  Wing root LE : {wing_root.xyz_le}")
print(f"  LE match     : {np.allclose(body_tip.xyz_le, wing_root.xyz_le, atol=1e-6)}")
print(f"  Chord match  : body={body_tip.chord:.4f}, wing={wing_root.chord:.4f} → "
      f"{'OK' if abs(body_tip.chord - wing_root.chord) < 1e-6 else 'MISMATCH'}")

## 5. Design Space Bounds

In [ ]:
print(f"{'Variable':<28s} {'Lower':>8s} {'Default':>8s} {'Upper':>8s}")
print("─" * 56)
d = {k: getattr(params, k) for k in VAR_NAMES}
for name in VAR_NAMES:
    lo, hi = BOUNDS[name]
    val = d[name]
    print(f"{name:<28s} {lo:8.3f} {val:8.3f} {hi:8.3f}")